# Train Requirement Classifier (BERT)

This notebook trains a BERT-based binary classifier to identify requirements in SRS documents.
It uses the aligned PURE dataset (`pure_train_aligned.json`) for training.

## 1. Setup and Imports

In [ ]:
!pip install transformers datasets scikit-learn evaluate accelerate torch

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Data

In [ ]:
# Define paths
base_dir = Path("..")
data_path = base_dir / "data" / "aligned" / "pure_train_aligned.json"

# Load data
with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame for easier handling
df = pd.DataFrame(data)
print(f"Total samples: {len(df)}")
print(df["label"].value_counts())

In [ ]:
# Prepare for Hugging Face Dataset
# We strictly need 'text' and 'label' columns
dataset_df = df[["text", "label"]].copy()

# Split into Train (80%) and Test (20%)
train_df, test_df = train_test_split(dataset_df, test_size=0.2, random_state=42, stratify=dataset_df["label"])

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(dataset)

## 3. Preprocessing (Tokenization)

In [ ]:
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 4. Model Initialization

In [ ]:
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

## 5. Training Configuration

In [ ]:
def compute_metrics(p):
    pred, labels = p
    pred = np.argmax(pred, axis=1)

    accuracy = accuracy_score(labels, pred)
    recall = recall_score(labels, pred)
    precision = precision_score(labels, pred)
    f1 = f1_score(labels, pred)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

output_dir = "../models/bert-classifier"

training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 6. Training Loop

In [ ]:
trainer.train()

## 7. Evaluation & Saving

In [ ]:
# Final Evaluation
metrics = trainer.evaluate()
print("Final Metrics:", metrics)

# Save Model
trainer.save_model(output_dir)
print(f"Model saved to {output_dir}")

In [ ]:
# Quick Test on Sample Text
text_samples = [
    "The system shall allow the user to login with a valid username and password.",
    "The quick brown fox jumps over the lazy dog.",
    "The application performance must typically be within 2 seconds."
]

inputs = tokenizer(text_samples, padding=True, truncation=True, return_tensors="pt").to(device)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)

for text, pred in zip(text_samples, predictions):
    label = "Requirement" if pred == 1 else "Not Requirement"
    print(f"Input: {text}")
    print(f"Prediction: {label}\n")